# ECG Denoising Pipeline Demo
Run the pipeline on the bundled example data.

In [2]:
# Demonstruoja triukšmų šalinimą iš  ECG signalų, naudojant ecg_denoising_pipeline.
# taip pat demonstruoja funkcijų, naudojamų aptiktų triukšmų parametrų skaičiavimui, darbą
# Pagrindinis tikslas - parodyti, kaip naudoti ecg_denoising_pipeline, o ne išsamiai analizuoti rezultatus.

# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
from datetime import datetime, timedelta
# "Run ECG denoising pipeline, ECG ectopy detecting and removing pipeline,
# HRV calculation on a single .npy ECG file.\n"

# from matplotlib.collections import LineCollection
from pathlib import Path
import math, time
# import numpy as np
# from typing import Dict, List, Tuple, Iterable
from ecg_denoising_pipeline.utils import map_mark_denoised_to_start
from ecg_denoising_pipeline.utils import convert_seconds_to_hms
from ecg_denoising_pipeline.io_utils import load_gaps, load_array_or_fail
from ecg_denoising_pipeline.config import load_denoising_config_yaml
import numpy as np
from typing import Tuple, List

from ecg_denoising_pipeline import DenoisingPipelineConfig
from ecg_denoising_pipeline import resolve_model_path
from ecg_denoising_pipeline import ECGDenoisingPipeline
from ecg_denoising_pipeline import DenoisingPipelineResult

# from ecg_denoising_pipeline.index_map import Interval
from record_noise_stats import calc_noise_stats_from_result


# --- ecg_denoising_pipeline imports (as in your original) ---
from ecg_denoising_pipeline import (
    print_heading,
    as_seconds,
)

from ecg_denoising_pipeline.steps import check_denoising_config

from denoising_util import (
    find_project_root_by_name,
    intervals_to_hms,
    samples_dict_to_seconds,
)

from typing import List, Tuple

Interval = Tuple[int, int]

def print_detailed_denoising_results(res_denoising: DenoisingPipelineResult, cfg_denoising: DenoisingPipelineConfig):
    
    print_heading("ECG DENOISING pipeline results")
    print(f"len_original: {len(res_denoising.ecg_orig)}")
    print(f"len_start   : {len(res_denoising.ecg_start)}") 
    print(f"len_denoised   : {len(res_denoising.ecg_denoised)}")

    print("\nMaps (sample intervals):")
    print("map_gaps    :", res_denoising.map_gaps)
    print("map_outliers:", res_denoising.map_outliers)
    print("map_rdropouts:", res_denoising.map_rdropouts)
    print("map_motions :", res_denoising.map_motions)

    print_heading("Detected intervals (in samples)")
    print("Outliers (start):", res_denoising.outliers_indices_start)
    print("Rdropouts (nout):", res_denoising.rdropouts_indices_nout)
    print("Motions (nrd)   :", res_denoising.motions_indices_nrd)

    print_heading("Detected intervals (in seconds)")
    print("Outliers (start):", as_seconds(res_denoising.outliers_indices_start, cfg_denoising.fs))
    print("Rdropouts (nout):", as_seconds(res_denoising.rdropouts_indices_nout, cfg_denoising.fs))
    print("Motions (nrd)   :", as_seconds(res_denoising.motions_indices_nrd, cfg_denoising.fs))

    # print_heading("Detected intervals (as HH:MM:SS, using fixed start_dt)")
    # print("Outliers (start):", intervals_to_hms(res_denoising.outliers_indices_start, cfg_denoising.fs, start_dt))
    # print("Rdropouts (nout):", intervals_to_hms(res_denoising.rdropouts_indices_nout, cfg_denoising.fs, start_dt))
    # print("Motions (nrd)   :", intervals_to_hms(res_denoising.motions_indices_nrd,   cfg_denoising.fs, start_dt))

    print("\nProjected intervals (in samples):")
    print("projected_to_orig", res_denoising.projected_to_orig)
    print("projected_to_start", res_denoising.projected_to_start)

    print("\nProjected intervals (in seconds):")
    proj_orig_sec  = samples_dict_to_seconds(res_denoising.projected_to_orig,  cfg_denoising.fs,ndigits=3)
    proj_start_sec = samples_dict_to_seconds(res_denoising.projected_to_start, cfg_denoising.fs,ndigits=3)

    print("projected_to_orig (s):", proj_orig_sec)
    print("projected_to_start (s):", proj_start_sec)


def prepare_denoising_pipeline(
    config_path: Path,
    model_dir: Path,
) -> DenoisingPipelineConfig:
    """Pipeline-aware runner that wires configuration, data loading and execution."""

    if not config_path.exists():
        raise FileNotFoundError(f"Config file not found: {config_path}")
    if not model_dir.exists():
        raise FileNotFoundError(f"Model directory not found: {model_dir}")

    # print_heading("Project paths")
    # print("DATA_DIR:", data_dir)
    # print("CONFIG  :", config_path)
    # print("MODEL_DIR:", model_dir)

    cfg = load_denoising_config_yaml(str(config_path))
    fs = float(cfg.fs)

    # print_heading("Denoising pipeline config")
    # friendly_print_denoising_cfg(cfg)
    check_denoising_config(cfg)
    
    # print_heading("Input data")
    # print(f"File: {file_name}")
    # len_secs = math.ceil(len(x) / fs)
    # h, m, s = convert_seconds_to_hms(len_secs)
    # print(f"len(ecg): {len(x)} samples (~{len_secs:.1f} s) | {h:02d}:{m:02d}:{s:02d}")

    # print("path:", path)
    # print(f"Loaded {len(gaps_indices)} gap intervals.")

    cfg.motions.model_name = resolve_model_path(model_dir, cfg.motions.model_name)
    cfg.motions.enabled = True

    # print_heading("UNet model")
    # print("UNet model path:", cfg.motions.model_name)
    # print("Motions enabled:", bool(getattr(cfg.motions, "enabled", True)))

    return cfg



start_time_1 = time.time()

print("\n**********TEST FOR ECG RECORD DENOISING\n")

HOME = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = find_project_root_by_name(target="PROJECT_TRAIN_UNET", start=HOME)

print("PROJECT ROOT_DIR:", PROJECT_ROOT)
print("PROJECT HOME DIR:", HOME)

print("\n*****Data, Configuration and Model Paths:")
cfg_denoising_path = PROJECT_ROOT/ 'CONFIG'/ 'denoising_config.yaml'
model_unet_dir = PROJECT_ROOT/'MODEL_UNET'
data_dir = PROJECT_ROOT/"DATA_ORIG"/"ecg_zive_npy"
gaps_dir = PROJECT_ROOT/'DATA/LONG_ECG_AND_SCRIPTS'

print("DATA_DIR:", data_dir)
print("CONFIG  :", cfg_denoising_path)
print("MODEL_DIR:", model_unet_dir)

# Preparing the Denoising pipeline with configuration and model paths
print("\n*****Denoising pipeline config:")
cfg_denoising = prepare_denoising_pipeline(cfg_denoising_path, model_unet_dir)
pipe = ECGDenoisingPipeline(cfg_denoising)

        #   INPUT DATA **************************************

# start0
# Input data
# Pseudo_annotated
fileNames = ['1019_118.npy'] #
fileNames = ['1001_4.npy'] #
fileNames = ['1005_2.npy'] #
fileNames = ['1008_1.npy'] #
fileNames = ['1008_1.npy','1008_10.npy'] #

fileNames = ['1008_10.npy'] #


for file_name in fileNames:
    print_heading("Input data")
    print(f"File: {file_name}")

    # Load ECG signal and gaps for display
    x = load_array_or_fail(data_dir, file_name)

    len_secs = math.ceil(len(x) / cfg_denoising.fs)
    h, m, s = convert_seconds_to_hms(len_secs)
    print(f"\nLoaded ECG signal: len(ecg): {len(x)} samples (~{len_secs:.1f} s) duration: {h:02d}:{m:02d}:{s:02d}")

    
            # DENOISING **************************************

    print("\nRunning Denoising pipeline...")
    # Execute the pipeline in the notebook with explicit arguments
    res_denoising = pipe.run(x, gaps_indices=[])

            # RESULTS **************************************

    print_detailed_denoising_results(res_denoising, cfg_denoising)

    print("\nCalculating noise stats from denoising results...")
    stats = calc_noise_stats_from_result(res_denoising)
    # print(stats["out"], stats["rdr"], stats["noi"], stats["tp_pct"])
    print(
        f"Noise stats → "
        f"\nOUT: {stats['out']}, "
        f"\nRDR: {stats['rdr']}, "
        f"\nNOI: {stats['noi']}, "
        f"\nTP%: {stats['tp_pct']:.1f}"
    )

end_time_1 = time.time()
time_taken_1 = end_time_1 - start_time_1
print(f"\nTime taken for the DENOISING pipeline: {time_taken_1:.2f} secs")  





**********TEST FOR ECG RECORD DENOISING

PROJECT ROOT_DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET
PROJECT HOME DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/0_SELECT_ZIVE_DATA_2023/UPDATE_RECORDS_LIST_ADDING_ML_NOISES

*****Data, Configuration and Model Paths:
DATA_DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/DATA_ORIG/ecg_zive_npy
CONFIG  : /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/CONFIG/denoising_config.yaml
MODEL_DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/MODEL_UNET

*****Denoising pipeline config:

Check if the conditions are met:
OK. 2*sliding_window_size is less than main_window_size
OK. main_window_size is less than all specified values: t_gap_max, extra_interval, t_start_gap_max, t_end_gap_max

Input data
----------
File: 1008_10.npy

Loaded ECG signal: len(ecg): 127999 samples (~640.0 s) duration: 00:10:40

Running Denoising pipeline...

ECG DENOISING pipeline results
------------------------------
len_original: 127999
len_start   : 127999
len_denois